# 05 · Development threshold and FP32 test evaluation

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya

**Priority:** 0 (Checkpoint 1)

**Purpose.** Select the FP32 EER threshold on development scores only, freeze it, then evaluate the test set once.

**In plain words.** A score turns into a decision with a cutoff: `score >= threshold` means bona fide. We choose the cutoff where development APCER and BPCER are closest. Only after it is saved do we look at the test set, and the code refuses to re-evaluate the test set with a different threshold.

**Expected inputs**
- Frozen checkpoint (notebook 04).
- Processed dev and test evaluation frames (notebook 02).

**Expected outputs**
- `QPAD_Project/results/fp32/{dev,test}_{frame,video}_scores.csv`
- `threshold.json`, `dev_threshold_candidates.csv`, `metrics.json`, `test_errors_by_*.csv`, figures
- Report assets in `reports/checkpoint1/{tables,figures}` (repository copy, aggregate results only)

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [ ]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

In [ ]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

In [ ]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

In [ ]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

In [ ]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = True                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42
CONFIRM_THRESHOLD_FROZEN = False   # full run: set True after inspecting the dev threshold
UPDATE_REPORT_ASSETS = True
if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb05", paths.logs / "nb05.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

## 1. Score development videos and select the threshold
Rule: minimise |APCER − BPCER|; ties → lowest ACER; still tied → higher threshold (more secure). Counts are compared with exact integers.

In [ ]:
import torch, pandas as pd
DEVICE = os.environ.get("QPAD_DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")
from qpad.pipeline import select_fp32_threshold
THR = select_fp32_threshold(paths, cfgs, DRY_RUN, DEVICE, QPAD_CODE_DIR, tqdm_progress("dev frames"))
t = THR["threshold"]
print(f"threshold = {t['threshold']:.6f}")
print(f"dev APCER = {t['apcer_errors']}/{t['n_attacks']} = {100 * t['apcer']:.2f}%")
print(f"dev BPCER = {t['bpcer_errors']}/{t['n_bona_fide']} = {100 * t['bpcer']:.2f}%")
print(f"dev ACER  = {100 * t['acer']:.2f}%")
display(THR["table"][THR["table"].selected | (THR["table"].abs_diff_key <= THR["table"].abs_diff_key.min())].head(10))

In [ ]:
from IPython.display import Image
for f in THR["figures"]:
    display(Image(filename=f["png"], width=640)); print(f["caption"])

## 2. Test evaluation (once)
The threshold file is hashed into a lock. Changing the threshold after this cell has run raises an error.

In [ ]:
from qpad.pipeline import evaluate_fp32_test
if not DRY_RUN and not CONFIRM_THRESHOLD_FROZEN:
    raise RuntimeError("Inspect the development threshold, then set CONFIRM_THRESHOLD_FROZEN = True.")
EVAL = evaluate_fp32_test(paths, cfgs, DRY_RUN, DEVICE, QPAD_CODE_DIR, progress=tqdm_progress("test frames"))
m = EVAL["metrics"]["test"]
for k in ("print_apcer", "replay_apcer", "apcer_pooled", "apcer_max_pai", "bpcer"):
    print(f"{k:14s} {m[k]['text']}")
print(f"ACER (pooled) {100 * m['acer']:.2f}%  | ACER (worst PAI, OULU convention) {100 * m['acer_max_pai']:.2f}%")
print(f"EER (test scores, descriptive) {100 * m['eer']['value']:.2f}%  | AUROC {m['auroc']:.4f}")
print("confusion:", m["confusion"])

In [ ]:
res_dir = Path(EVAL["results_dir"])
for by in ("phone", "session", "phone_session"):
    print("errors by", by); display(pd.read_csv(res_dir / f"test_errors_by_{by}.csv"))
for f in EVAL["figures"]:
    display(Image(filename=f["png"], width=560)); print(f["caption"])

## 3. Sanity reference (not a target)
George & Marcel (ICB 2019, Table 2) report DeepPixBiS on OULU-NPU Protocol 2 as APCER 11.39 %, BPCER 0.56 %, ACER 5.97 %, with worst-PAI APCER and a development-EER threshold. A very different result means: inspect preprocessing, labels, loss, sampling and score aggregation before continuing.

In [ ]:
REFERENCE_PUBLISHED = {"apcer_max_pai_pct": 11.39, "bpcer_pct": 0.56, "acer_max_pai_pct": 5.97}   # George & Marcel 2019, Table 2
ours = {"apcer_max_pai_pct": m["apcer_max_pai"]["percent"], "bpcer_pct": m["bpcer"]["percent"],
        "acer_max_pai_pct": 100 * m["acer_max_pai"]}
display(pd.DataFrame([REFERENCE_PUBLISHED, ours], index=["DeepPixBiS (ICB 2019)", "this run"]))

In [ ]:
# --- Report assets (aggregate results only; no images of faces) --------------------------------
from qpad.reporting import build_report_assets
from qpad.reproducibility import git_commit
if UPDATE_REPORT_ASSETS and not DRY_RUN:
    w = build_report_assets(OUT.results, OUT.reports, QPAD_CODE_DIR / "reports" / "checkpoint1", git_commit(QPAD_CODE_DIR))
    print(json.dumps(w, indent=1))
else:
    print("dry run: report assets not updated")

In [ ]:
# --- Final summary -----------------------------------------------------------------------------
print("threshold:", res_dir / "threshold.json")
print("metrics  :", res_dir / "metrics.json")
print("lock     :", res_dir / "test_evaluation_lock.json")

## Before you continue — inspect

- [ ] The threshold came from development scores only (`threshold.json → split = dev`).
- [ ] Every rate is shown with numerator/denominator.
- [ ] Error breakdown by phone/session/attack type is plausible (no single phone dominating errors).
- [ ] Compare with the sanity reference; investigate large gaps before Priority 1.
- [ ] Commit code, then rebuild report assets so the report carries the right Git commit.

**Next notebook:** `06_export_onnx_tensorrt_validation.ipynb`